## Bias in LLMs

# 🧠 What’s Really Happening Behind the Scenes (Hugging Face Transformers)

When you run:

```python
from transformers import pipeline

classifier = pipeline("sentiment-analysis")
classifier([
    "I've been waiting for a HuggingFace course my whole life.",
    "I hate this so much!",
])
```

and get:

```python
[
  {'label': 'POSITIVE', 'score': 0.96},
  {'label': 'NEGATIVE', 'score': 0.999}
]
```

— here’s what’s quietly going on under the hood 👇

---

## ⚙️ 1. The Pipeline = 3 Hidden Steps

Every pipeline in 🤗 Transformers secretly runs through these steps:

```
preprocessing → model (inference) → postprocessing
```

So when you call `pipeline("sentiment-analysis")`, it’s basically doing *all the setup work* for you — tokenizing, running the model, and converting the results into readable labels.

---

## ✂️ 2. Preprocessing (Tokenizing the Text)

Transformers can’t read raw text — they need numbers.
So a **tokenizer** does the job of turning your text into a list of integers.

It:

* Breaks text into tokens (words, subwords, or punctuation)
* Maps each token to an integer ID
* Adds helpful stuff like attention masks

Example:

```python
from transformers import AutoTokenizer

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

inputs = tokenizer(
    ["I love AI.", "I hate bugs."],
    padding=True,
    truncation=True,
    return_tensors="pt"
)
```

Output looks like this (simplified):

```python
{
  'input_ids': tensor([[101, 1045, 2293, 993, 102],
                       [101, 1045, 5223, 7398, 102]]),
  'attention_mask': tensor([[1, 1, 1, 1, 1],
                            [1, 1, 1, 1, 1]])
}
```

✅ **`input_ids`** = token numbers
✅ **`attention_mask`** = 1 for real tokens, 0 for padding

---

## 📦 3. Tensors (The Model’s Favorite Food)

The model eats **tensors**, not lists.
They’re like NumPy arrays — just n-dimensional boxes of numbers.

```python
return_tensors="pt"  # means PyTorch tensors
```

Tensors have shapes like `[batch_size, sequence_length]`.

---

## 🧩 4. The Model (Transformer Backbone)

You can load the pretrained model like this:

```python
from transformers import AutoModel
model = AutoModel.from_pretrained(checkpoint)
```

This gives you the **base Transformer**, which outputs **hidden states** — big 3D vectors that represent meaning for every token.

Example shape:

```
[batch_size, sequence_length, hidden_size]
[2, 16, 768]
```

It’s like:

> “For each word, here’s a 768-dimensional thought vector describing its meaning.”

---

## 🎯 5. The Model Head (Task-Specific Part)

Hidden states are cool, but they’re just raw understanding.
To *do* something (like classify sentiment), we attach a **head** on top of the model.

For sentiment analysis:

```python
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
outputs = model(**inputs)
```

Now the output is smaller and task-focused:

```
[batch_size, num_labels] → [2, 2]
```

Those numbers are called **logits** — raw scores for each label.

---

## 🔢 6. Postprocessing (Turning Logits → Probabilities → Labels)

The logits aren’t probabilities yet, so we use **SoftMax** to fix that:

```python
import torch

preds = torch.nn.functional.softmax(outputs.logits, dim=-1)
print(preds)
```

Example output:

```
tensor([[0.04, 0.96],
        [0.999, 0.0005]])
```

Finally, we map those to human-readable labels:

```python
model.config.id2label
# {0: 'NEGATIVE', 1: 'POSITIVE'}
```

✅ Sentence 1 → POSITIVE (0.96)
✅ Sentence 2 → NEGATIVE (0.999)

Boom — that’s your final answer!

---

## 💡 7. Tiny Things Worth Remembering

* **Padding & truncation** make all sentences the same length
* **Attention mask** tells the model which tokens are padding
* **Always** use the same checkpoint for model + tokenizer
* The `pipeline()` function is just a shortcut that does all this for you

---

## 🧾 Quick Command Cheat-Sheet

```python
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"

# 1️⃣ Tokenize
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
inputs = tokenizer(["I love X", "I hate Y"], padding=True, truncation=True, return_tensors="pt")

# 2️⃣ Load Model
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

# 3️⃣ Run
outputs = model(**inputs)

# 4️⃣ Convert to Probabilities
probs = torch.nn.functional.softmax(outputs.logits, dim=-1)

# 5️⃣ Map to Labels
labels = model.config.id2label
```

In [1]:
from transformers import pipeline

unmasker = pipeline("fill-mask", model="bert-base-uncased")
result = unmasker("This man works as a [MASK].")
print([r["token_str"] for r in result])

result = unmasker("This woman works as a [MASK].")
print([r["token_str"] for r in result])

d:\Bishal_Shrestha\Machine_Learning\Natural_Language_processing\NLP_Expertise\.venv_asr_sentiment\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification 

['carpenter', 'lawyer', 'farmer', 'businessman', 'doctor']
['nurse', 'maid', 'teacher', 'waitress', 'prostitute']


## pipeline function behind the bars

In [2]:
from transformers import AutoTokenizer

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

raw_inputs = [
    "I've been waiting for a HuggingFace course my whole life.",
    "I hate this so much!",
]
inputs = tokenizer(raw_inputs, padding=True, truncation=True, return_tensors="pt")
print(inputs)



{'input_ids': tensor([[  101,  1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,
          2607,  2026,  2878,  2166,  1012,   102],
        [  101,  1045,  5223,  2023,  2061,  2172,   999,   102,     0,     0,
             0,     0,     0,     0,     0,     0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]])}


In [3]:
from transformers import AutoModel

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModel.from_pretrained(checkpoint)
outputs = model(**inputs)
print(outputs.last_hidden_state.shape)

torch.Size([2, 16, 768])


In [5]:
from transformers import AutoModelForSequenceClassification
checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
outputs = model(**inputs)
print(outputs.logits)

tensor([[-1.5607,  1.6123],
        [ 4.1692, -3.3464]], grad_fn=<AddmmBackward0>)


In [6]:
import torch
predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
print(predictions)

tensor([[4.0195e-02, 9.5980e-01],
        [9.9946e-01, 5.4418e-04]], grad_fn=<SoftmaxBackward0>)


In [7]:
model.config.id2label

{0: 'NEGATIVE', 1: 'POSITIVE'}

In [11]:
from transformers import AutoModel
from transformers import AutoConfig

bert_model = AutoModel.from_pretrained("bert-base-cased")
print(bert_model)
gpt_model = AutoModel.from_pretrained("gpt2")
print(gpt_model)
bart_model = AutoModel.from_pretrained("facebook/bart-base")
print(bart_model)

d:\Bishal_Shrestha\Machine_Learning\Natural_Language_processing\NLP_Expertise\.venv_asr_sentiment\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\bisha\.cache\huggingface\hub\models--bert-base-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' p

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(28996, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

d:\Bishal_Shrestha\Machine_Learning\Natural_Language_processing\NLP_Expertise\.venv_asr_sentiment\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\bisha\.cache\huggingface\hub\models--facebook--bart-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xe

BartModel(
  (shared): BartScaledWordEmbedding(50265, 768, padding_idx=1)
  (encoder): BartEncoder(
    (embed_tokens): BartScaledWordEmbedding(50265, 768, padding_idx=1)
    (embed_positions): BartLearnedPositionalEmbedding(1026, 768)
    (layers): ModuleList(
      (0-5): 6 x BartEncoderLayer(
        (self_attn): BartAttention(
          (k_proj): Linear(in_features=768, out_features=768, bias=True)
          (v_proj): Linear(in_features=768, out_features=768, bias=True)
          (q_proj): Linear(in_features=768, out_features=768, bias=True)
          (out_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (activation_fn): GELUActivation()
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (fc2): Linear(in_features=3072, out_features=768, bias=True)
        (final_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      )
    )
 

In [13]:
bert_config = AutoConfig.from_pretrained("bert-base-cased")
print(type(bert_config))
bart_config = AutoConfig.from_pretrained("facebook/bart-base")
print(type(bart_config))
gpt_config = AutoConfig.from_pretrained("gpt2")
print(type(gpt_config))

<class 'transformers.models.bert.configuration_bert.BertConfig'>
<class 'transformers.models.bart.configuration_bart.BartConfig'>
<class 'transformers.models.gpt2.configuration_gpt2.GPT2Config'>


In [14]:
from transformers import BertConfig
bert_config = BertConfig.from_pretrained("bert-base-cased")
print(bert_config)

BertConfig {
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.57.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 28996
}



In [17]:
bert_config = BertConfig.from_pretrained("bert-base-cased",num_hidden_layers=10)
bert_model = BertConfig(bert_config)

In [18]:
bert_model.save_pretrained("my_bert_model")

In [ ]:
from transformers import BertModel
bert_model = BertModel.from_pretrained("my_bert_model")

### Sentences
   ↓
### Tokenizer (convert text → numbers)
   ↓
### Model Forward Pass (run through DistilBERT)
   ↓
### Logits (raw prediction scores)
   ↓
### Softmax (convert scores → probabilities)
   ↓
### Label Mapping (find the label with highest probability)
   ↓
### Output: [{"label": ..., "score": ...}, ...]


In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

sentences = ["I've been waiting for a HuggingFace course my whole life.",
             "I hate this so much!"]

# 1) tokenize to tensors with padding/truncation
inputs = tokenizer(sentences, padding=True, truncation=True, return_tensors="pt")

# 2) forward pass
outputs = model(**inputs)   # outputs.logits shape [2, 2]

# 3) logits -> probabilities
probs = torch.nn.functional.softmax(outputs.logits, dim=-1)

# 4) map ids to labels
id2label = model.config.id2label  # e.g., {0: 'NEGATIVE', 1: 'POSITIVE'}
predicted = [{"label": id2label[int(p.argmax())], "score": float(p.max())} for p in probs]

print(predicted)


d:\Bishal_Shrestha\Machine_Learning\Natural_Language_processing\NLP_Expertise\.venv_asr_sentiment\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[{'label': 'POSITIVE', 'score': 0.9598049521446228}, {'label': 'NEGATIVE', 'score': 0.9994558691978455}]


In [3]:
model.config.id2label

{0: 'NEGATIVE', 1: 'POSITIVE'}